In [15]:
import pandas as pd
import numpy as np
import os

In [16]:
severity_rank = {
    "(X)": 1,
    "(NI0)": 2,
    "(CT)": 3,
    "(M03)": 4,
    "(M2)": 5,
    "(M1)": 6,
    "(CO3)": 7,
    "(TCX)": 8,
    "(F7)": 9,
    "(F6)": 10,
    "(F5)": 11,
    "(F3)": 12,
    "(F2)": 13,
    "(F1)": 14
}

In [17]:
def prepare_dataset(data_violent_filt):

    data_cleaned = data_violent_filt[[
    "sex", "age", "race",
    "priors_count", "juv_fel_count",
    "juv_misd_count", "juv_other_count","decile_score",
    "c_jail_in", "c_jail_out", "is_recid"
    ]].copy()
    data_cleaned['c_jail_in'] = pd.to_datetime(data_cleaned['c_jail_in'], format="%d/%m/%Y %H:%M")
    data_cleaned['c_jail_out'] = pd.to_datetime(data_cleaned['c_jail_out'], format="%d/%m/%Y %H:%M")
    data_cleaned['jail_time'] = (data_cleaned['c_jail_out'] - data_cleaned['c_jail_in']).dt.total_seconds() / (3600)
    data_cleaned.drop(columns=['c_jail_in', 'c_jail_out'], inplace=True)
    # list(data_cleaned['c_charge_degree'].unique())
    data_cleaned = data_cleaned.map(lambda x: severity_rank.get(x) if x in severity_rank else x)
    data_cleaned.dropna(inplace=True)
    return data_cleaned

In [ ]:

def get_random_partitions(dataset, num_partitions):

    ds_random = dataset.sample(frac=1, random_state=42).reset_index(drop=True)
    indices = ds_random.index
    split_indices = np.array_split(indices, num_partitions)
    return [ds_random.loc[idx] for idx in split_indices]



def create_federated_dataset_random(num_clients=5):
    """
    With write=True, this function writes the dataset, otherwise it simply 
    """
    dataset = pd.read_csv('../data/cox-violent-parsed_filt.csv')
    dataset = prepare_dataset(dataset)
    os.makedirs('../federated_data_random', exist_ok=True)

    data_partitions = get_random_partitions(dataset,num_clients+1)

    for i,partition in enumerate(data_partitions):

        if(i == 0):
            partition.to_csv(f'../federated_data_random/client_random_test.csv', index=False) # type: ignore
            continue

        partition.to_csv(f'../federated_data_random/client_random_{i}.csv', index=False) # type: ignore

    return data_partitions

#### Centralized Dataset

In [19]:
data_violent_filt = pd.read_csv('../data/cox-violent-parsed_filt.csv')
data_prepared = prepare_dataset(data_violent_filt)
data_prepared.to_csv('../data/centralized_dataset.csv', index=False)


#### Decenralized Dataset

In [ ]:
create_federated_dataset_random()

Client 1 size: 3489 | Race distribution:
race
Caucasian           0.452565
Hispanic            0.262253
African-American    0.231012
Other               0.054170
Name: proportion, dtype: float64

Client 2 size: 1439 | Race distribution:
race
African-American    0.729673
Caucasian           0.159138
Other               0.069493
Native American     0.025017
Asian               0.009729
Hispanic            0.006949
Name: proportion, dtype: float64

Client 3 size: 3153 | Race distribution:
race
Caucasian           0.651126
Other               0.155408
Hispanic            0.135427
African-American    0.036790
Asian               0.017444
Native American     0.003806
Name: proportion, dtype: float64

Client 4 size: 2981 | Race distribution:
race
African-American    0.790003
Caucasian           0.171754
Other               0.026837
Hispanic            0.010735
Native American     0.000671
Name: proportion, dtype: float64

Client 5 size: 7254 | Race distribution:
race
African-American    0.753

[[np.int64(4656),
  np.int64(2070),
  np.int64(8147),
  np.int64(2264),
  np.int64(14527),
  np.int64(7780),
  np.int64(15739),
  np.int64(14492),
  np.int64(9663),
  np.int64(11504),
  np.int64(7635),
  np.int64(439),
  np.int64(434),
  np.int64(16717),
  np.int64(595),
  np.int64(3522),
  np.int64(13219),
  np.int64(13103),
  np.int64(3003),
  np.int64(8655),
  np.int64(9797),
  np.int64(5866),
  np.int64(16930),
  np.int64(8993),
  np.int64(4026),
  np.int64(15677),
  np.int64(8286),
  np.int64(2988),
  np.int64(11436),
  np.int64(2879),
  np.int64(12775),
  np.int64(7106),
  np.int64(1923),
  np.int64(4723),
  np.int64(2621),
  np.int64(10350),
  np.int64(10079),
  np.int64(10461),
  np.int64(5032),
  np.int64(15452),
  np.int64(1701),
  np.int64(12164),
  np.int64(12820),
  np.int64(7308),
  np.int64(14661),
  np.int64(16197),
  np.int64(16869),
  np.int64(10),
  np.int64(5627),
  np.int64(13958),
  np.int64(4051),
  np.int64(14529),
  np.int64(13475),
  np.int64(14282),
  np.int6

In [5]:
from get_datasets import get_federated_datasets_random, get_federated_datasets_sensitive, get_centralized_dataset

federated_random = get_federated_datasets_random()
federated_sensitive = get_federated_datasets_sensitive()
centralized_dataset = get_centralized_dataset()

federated_random[0]

,id,name,first,last,sex,dob,age,age_cat,race,juv_fel_count,...,vr_charge_desc,type_of_assessment,decile_score.1,score_text,screening_date,v_type_of_assessment,v_decile_score,v_score_text,priors_count.1,event
0,8242.0,dylan welly,dylan,welly,Male,18/04/1994,22,Less than 25,Caucasian,0,...,NaN,Risk of Recidivism,4,Low,16/03/2013,Risk of Violence,5,Medium,0,0
1,6933.0,guillermo laffiteau,guillermo,laffiteau,Male,23/05/1957,58,Greater than 45,Hispanic,0,...,NaN,Risk of Recidivism,1,Low,20/06/2014,Risk of Violence,1,Low,4,0
2,9844.0,luis gonzalez,luis,gonzalez,Male,22/05/1992,23,Less than 25,Caucasian,0,...,NaN,Risk of Recidivism,2,Low,26/12/2013,Risk of Violence,4,Low,0,0
3,NaN,joseph ambers,joseph,ambers,Male,17/09/1966,49,Greater than 45,Caucasian,0,...,NaN,Risk of Recidivism,7,Medium,02/08/2014,Risk of Violence,1,Low,6,0
4,NaN,anthony nicholasi,anthony,nicholasi,Male,15/11/1993,22,Less than 25,Caucasian,0,...,NaN,Risk of Recidivism,2,Low,01/07/2014,Risk of Violence,4,Low,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3659,3849.0,jerrod moore,jerrod,moore,Male,15/11/1983,32,25 - 45,African-American,0,...,NaN,Risk of Recidivism,8,High,15/01/2013,Risk of Violence,6,Medium,12,0
3660,10599.0,manuel aguasvivas,manuel,aguasvivas,Male,06/12/1988,27,25 - 45,Caucasian,0,...,NaN,Risk of Recidivism,7,Medium,17/09/2014,Risk of Violence,6,Medium,2,0
3661,6226.0,lawrence gaston,lawrence,gaston,Male,08/04/1993,23,Less than 25,African-American,0,...,NaN,Risk of Recidivism,4,Low,26/08/2013,Risk of Violence,6,Medium,1,0
3662,NaN,eric mckenzie,eric,mckenzie,Male,15/03/1990,26,25 - 45,African-American,0,...,NaN,Risk of Recidivism,9,High,26/08/2013,Risk of Violence,10,High,0,0
